In [ ]:
import os
import sys

import tensorflow as tf
import yaml

sys.path.append(os.path.abspath("../"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import glob

import matplotlib.pyplot as plt
import numpy as np
import sklearn

from train.models import FullModel
from util import dataset as u_dataset
from util import keypoint as u_keypoint
from util import metrics as u_metrics

In [ ]:
batch_size = 1840

model_name = "20251105-144619.keras"
path_to_models = "../../models/finished/"

In [ ]:
# Get encoder config
def load_config(config_path):
    with open(config_path) as f:
        config = yaml.safe_load(f)
    return config


config_dir = f"../../logs/fit/{model_name.split('.')[0]}/"
config = load_config(config_dir + "config.yaml")

encoder_architecture = config["model"]["encoder"]["architecture"]
classifier_architecture = config["model"]["classifier"]["architecture"]
categories_config = config["categories"]

#### Load Test Data


In [ ]:
path_to_test_data = glob.glob("../../data/tfrecords/test_ds*.tfrecords")

test_ds = u_dataset.get_dataset(path_to_test_data)
test_ds = test_ds.batch(batch_size, drop_remainder=False)

#### Load Model


In [ ]:
model = FullModel.load(
    encoder_architecture,
    classifier_architecture,
    input_dims=config["model"]["encoder"]["input_dims"],
    filepath=path_to_models,
    filename=model_name,
    n_context=config["model"]["encoder"]["n_context"],
    only_train_encoder=config["model"]["encoder"]["only_train_encoder"],
    classifier_offsets=config["model"]["classifier"]["with_offsets"],
    encoder_only=False,
    verbose=True,
    n_meta=config["model"]["classifier"]["n_meta"],
    encoder_use_batch_norm=config["model"]["encoder"]["use_batch_norm"],
    classifier_use_batch_norm=config["model"]["classifier"]["use_batch_norm"],
    categories_config=config["categories"],
)
model.compile(optimizer=tf.keras.optimizers.Adam(), jit_compile=False)


#### Predict Data


In [ ]:
groundtruth_dataset = []
for x in test_ds:
    groundtruth_dataset.append(x)

# Take only image, camera, intrinsics for input
input_data = test_ds.map(lambda x: (x["image"], x["camera"], x["intrinsics"]))

# Convert tf.Data.Dataset into list
input_list = []
for x, y, z in input_data:
    input_list.append((x, y, z))

model.run_eagerly = True

predictions = [model.predict(x=batch) for batch in input_list]
metrics_list = [model.evaluate(x=batch, return_dict=True) for batch in groundtruth_dataset]

#### Evaluate Models


In [ ]:
# Calculate mean for metrics over each batch
keys = metrics_list[0].keys()
stacked_values = {key: tf.stack([metrics[key] for metrics in metrics_list]) for key in keys}
metrics_mean = {key: float(tf.reduce_mean(stacked_values[key])) for key in keys}

# Print Metrics
for key, value in metrics_mean.items():
    print(f"{key}: {value:.4f}")

### Accuracy, Precision, Recall


In [ ]:
def reshape_tensors(object_tensors, reshape=True):
    # object_tensors = [tensor[object_name] for tensor in tensors]

    stacked_tensors = {
        key: tf.stack([object_tensor[key] for object_tensor in object_tensors])
        for key in object_tensors[0]
    }
    if reshape:
        return {
            key: tf.reshape(tensor, (-1, *tensor.shape[2:]))
            for key, tensor in stacked_tensors.items()
        }
    else:
        return stacked_tensors

In [ ]:
groundtruth_reshaped = reshape_tensors([batch["ball"] for batch in groundtruth_dataset], True)
ball_predictions_reshaped = reshape_tensors(
    [batch["results"]["ball"] for batch in predictions], True
)
thresholds = np.linspace(0, 1, 40, endpoint=True)


conf_values_cla = [
    u_metrics.calculate_binary_metrics(ball_predictions_reshaped, groundtruth_reshaped, 0, cla)
    for cla in thresholds
]
# conf_values_enc = [
#     get_conf_matrix(ball_predictions_reshaped, groundtruth_reshaped, enc, 0) for enc in thresholds
# ]

In [ ]:
classifier_threshold = 0.7
encoder_threshold = 0

In [ ]:
precision_cla = [x["precision"] for x in conf_values_cla]
recall_cla = [x["recall"] for x in conf_values_cla]

plt.plot(thresholds, precision_cla, label="Precision")
plt.plot(thresholds, recall_cla, label="Recall")
plt.title("Precision/Recall Curve for Classifier Threshold")
plt.axvline(classifier_threshold, ls="--", color="r")
plt.xlabel("Threshold")
plt.ylabel("Precision/Recall")
plt.legend()
plt.show()

In [ ]:
a = u_metrics.calculate_binary_metrics(
    ball_predictions_reshaped, groundtruth_reshaped, encoder_threshold, classifier_threshold
)

In [ ]:
confusion_matrix = a["confusion_matrix"]

tf.print("Precision: ", a["precision"])
tf.print("Recall: ", a["recall"])
tf.print("fp rate:", confusion_matrix[0][1] / (confusion_matrix[0][1] + confusion_matrix[1][1]))
tf.print("fn rate: ", confusion_matrix[1][0] / (confusion_matrix[1][0] + confusion_matrix[0][0]))

sklearn.metrics.ConfusionMatrixDisplay(confusion_matrix).plot()

#### Write Metrics into a log file


In [ ]:
metrics_log = {
    "metrics": metrics_mean,
    "evaluation": {
        "encoder_threshold": encoder_threshold,
        "classifier_threshold": classifier_threshold,
        "precision": float(a["precision"]),
        "recall": float(a["recall"]),
        "confusion_matrix": (a["confusion_matrix"]).tolist(),
        "fp_rate": float(
            confusion_matrix[0][1] / (confusion_matrix[0][1] + confusion_matrix[1][1])
        ),
        "fn_rate": float(
            confusion_matrix[1][0] / (confusion_matrix[1][0] + confusion_matrix[0][0])
        ),
    },
    "fp_indices": a["fp_indices"].tolist(),
    "fn_indices": a["fn_indices"].tolist(),
}

log_file = config_dir + "metrics.yaml"
with open(log_file, "w") as f:
    # dump config file into log
    yaml.dump(metrics_log, f, sort_keys=False, default_flow_style=False)

print(f"Configuration logged to {log_file}")